# 3. Dispatch and execution

A run describes all planned work. A dispatch says which part to execute now and under which dependency policy. This is what makes partial execution, retries, resumption, and cancellation explicit rather than hidden worker behavior.


## Selection and dependency policy

Selections can target all tasks, node names, record keys, labels, or combinations. The planner converts that selection into task IDs. `INCLUDE_MISSING_UPSTREAM` closes over prerequisites that have not succeeded; `SELECTED_ONLY` keeps the dispatch limited to the selected tasks, so unresolved prerequisites can leave work blocked. Already reusable/succeeded work can be excluded.


In [ ]:
from provium_pipeline.dispatch.models import (
    DependencyPolicy,
    DispatchState,
    TaskSelection,
)

whole_run = TaskSelection(all=True)
one_node = TaskSelection(nodes=('tokenize',))
assert whole_run.all is True and one_node.nodes == ('tokenize',)
assert (
    DependencyPolicy.INCLUDE_MISSING_UPSTREAM
    != DependencyPolicy.SELECTED_ONLY
)
sorted(state.value for state in DispatchState)


## Worker loop and leases

`LocalRunExecutor` creates a dispatch and invokes a worker. `SerialDispatchWorker` repeatedly claims ready tasks, obtains an attempt lease, builds a frozen invocation, executes through Provium core, imports outputs, publishes output mappings, and transitions task/dispatch/run state.

A lease gives one worker temporary ownership. Its token prevents a stale worker from committing after ownership has moved. Heartbeats renew long attempts. Expired leases can be recovered. Retry policy computes bounded backoff and creates a new attempt without changing task identity.


In [ ]:
from provium_pipeline.execution.attempts import RetryPolicy, TaskAttemptLease
from provium_pipeline.execution.cancellation import CancellationToken
from provium_pipeline.dispatch.worker import SerialDispatchWorker
from provium_pipeline.execution.local_attempt import LocalTaskAttemptExecutor
from provium_pipeline.execution.run_executor import LocalRunExecutor

execution_parts = (
    TaskAttemptLease, RetryPolicy, CancellationToken,
    SerialDispatchWorker, LocalRunExecutor, LocalTaskAttemptExecutor,
)
assert all(execution_parts)
execution_parts


## Frozen invocation and output publication

`FrozenTaskInvocationBuilder` resolves every `$inputs` and `$nodes` binding from the stored snapshot and stored upstream outputs. The task executor materializes managed inputs into an isolated workspace, calls the standard Provium execution API, imports produced artifacts into managed storage, records output identities atomically, and cleans the workspace.

Prepared invocation caching avoids rebuilding the same invocation inside a worker. Computation caching is different: it reuses previously published outputs when procedure contract, configuration, and input identities produce the same computation key.


## Operational commands

- `provium run execute RUN_ID` creates and works a dispatch.
- Add `--node`, `--record`, or label filters for partial work.
- `provium dispatch wait DISPATCH_ID` observes a dispatch to a terminal state.
- `provium dispatch retry DISPATCH_ID` schedules eligible failed work.
- `provium run cancel RUN_ID` records cancellation and signals active local work.

**What to notice:** the state machine is the coordination protocol. The worker is intentionally replaceable. Next: [artifacts, cache, and GC](04-artifacts-cache-and-gc.ipynb). Reference: [execution](../docs/execution.md).
